
# 📊 Notebook 5 — Classification Pipeline: 15 Bước Đầy Đủ

**Bài giảng mẫu được tích hợp vào dự án dự đoán tái mua hàng (Repurchase Prediction)**

---

| Notebook | Vai trò |
|----------|---------|
| 1. Data Generation | ✅ Hoàn thành |
| 2. EDA | ✅ Hoàn thành |
| 3. Model Building | ✅ Hoàn thành — LR / XGBoost / LightGBM |
| 4. Model Evaluation | ✅ Hoàn thành — So sánh & đánh giá |
| **📌 5. Classification Pipeline 15 Bước** | **▶ Đang học — Pipeline đầy đủ theo chuẩn thầy** |

---

## 📋 Tổng quan 15 bước ML Pipeline

| Bước | Tên | Mô tả |
|:--:|------|------|
| 1 | Đặt vấn đề | Phân tích nghiệp vụ, pain point, KPI |
| 2 | Thu thập dữ liệu | Load data, hiểu nguồn, schema |
| 3 | Tiền xử lý | Missing, outliers, encoding, target dist |
| 4 | Định nghĩa Label/Target | Chọn biến mục tiêu, class imbalance |
| 5 | Feature Engineering | Tạo 15 features mới (7 cũ + 8 mới) |
| 6 | Feature Selection | Lọc correlation > 0.9 |
| 7 | Chia dữ liệu | Train/Test split stratified + Scale |
| 8 | Chọn mô hình | Logistic Regression → Random Forest → XGBoost |
| 9 | Huấn luyện | model.fit() |
| 10 | Đánh giá | Metrics + ROC + Feature Importance |
| 11 | Feature Reduction | Loại features importance < 0.01 |
| 12 | Tinh chỉnh | GridSearchCV |
| 13 | Retrain | Train lại với best params |
| 14 | Loop KPI | Kiểm tra F1 ≥ 0.72, AUC ≥ 0.87 |
| 15 | Cutoff Threshold | Tìm ngưỡng xác suất tối ưu |

> 💡 **Ghi nhớ:** Pipeline này áp dụng bài giảng mẫu vào bài toán thực tế: dự đoán khách hàng tái mua — dữ liệu e-commerce Việt Nam, 61,728 giao dịch.


In [ ]:

# ============================================================
# SECTION 0: Import thư viện và thiết lập
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# --- Thiết lập ---
RANDOM_STATE = 42
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'        : 11,
    'axes.grid'        : True,
    'grid.alpha'       : 0.2,
})

# --- Đường dẫn ---
BASE_DIR   = Path('.').resolve()
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
DATA_SRC   = Path(r'C:\Users\ADMIN\MOCK PROJECT VTI\pred_repurchase_dataset.csv')

# Fallback: nếu chưa có synthetic_data.csv → đọc từ nguồn gốc
DATA_PATH  = DATA_DIR / 'synthetic_data.csv' if (DATA_DIR / 'synthetic_data.csv').exists() else DATA_SRC

print("✅ Import thành công!")
print(f"   Data path : {DATA_PATH}")
print(f"   Exists?   : {DATA_PATH.exists()}")



---
# Bước 1: Đặt vấn đề

**Bối cảnh nghiệp vụ:**
- Công ty e-commerce cần **dự đoán khách hàng có tái mua hàng không** sau lần đầu
- Hiện tại team marketing gửi voucher đại trà → tỷ lệ chuyển đổi thấp, lãng phí ngân sách

**Pain point:**
- 67% khách không quay lại → mất chi phí acquisition cao
- Gửi voucher nhầm cho người không tái mua → lãng phí ~30,000 VND/người
- Bỏ sót khách có khả năng tái mua → mất cơ hội upsell ~150,000 VND revenue/người

**Mục tiêu ML:**
- Xây dựng model phân loại: **Tái mua (1)** vs **Không tái mua (0)**
- KPI mong muốn: **F1-Score ≥ 0.72**, **AUC ≥ 0.87**

> 💡 Trong bài toán CRM: **Cân bằng Recall và Precision** — bỏ sót khách tái mua (FN) tốn nhiều hơn gửi nhầm voucher (FP), nên ưu tiên F1 và Recall ≥ 0.70



---
# Bước 2: Thu thập dữ liệu

**Nguồn**: pred_repurchase_dataset.csv — Dữ liệu e-commerce Việt Nam (synthetic, production-ready format)
- Thu thập từ hành vi mua sắm của 61,728 khách hàng
- 20 cột: demographics, behavioral signals, aggregated purchase stats
- Nhãn `repurchase`: Otsu thresholding trên `purchase_count` → T* = 4.02 → label=1 nếu purchase_count ≥ 5

**Cột bị loại bỏ (anti-leakage):**
- `purchase_count`: Trực tiếp xác định nhãn → phải xóa trước khi dùng features
- `total_interactions`: = view + click + cart + wishlist + **purchase_count** → chứa purchase_count → **leaky**

> ⚠️ Dữ liệu thực tế: Có thể từ DB, API, Data Warehouse. Bước này quan trọng — schema phải được review kỹ trước khi dùng


In [ ]:

# === LOAD DỮ LIỆU ===
df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

# Loại cột leaky ngay khi load
LEAKY_COLS = ['total_interactions']     # = view+click+cart+wishlist+purchase_count
LEAKY_COLS = [c for c in LEAKY_COLS if c in df_raw.columns]
df = df_raw.drop(columns=LEAKY_COLS, errors='ignore')

print(f"Kích thước gốc : {df_raw.shape[0]:,} dòng × {df_raw.shape[1]} cột")
print(f"Sau loại leaky : {df.shape[0]:,} dòng × {df.shape[1]} cột")
if LEAKY_COLS:
    print(f"  Đã loại ({len(LEAKY_COLS)}): {LEAKY_COLS}")
print()
print("Danh sách cột:")
for i, col in enumerate(df.columns, 1):
    dtype = str(df[col].dtype)
    print(f"  {i:2d}. {col:<30} dtype={dtype}")
df.head(3)



---
# Bước 3: Tiền xử lý dữ liệu

**3 nhiệm vụ chính:**
1. Missing values → impute hoặc drop
2. Duplicates → kiểm tra và xử lý
3. Target distribution → hiểu class imbalance


In [ ]:

# === 3.1: Kiểm tra Missing Values ===
print("=" * 55)
print("3.1 MISSING VALUES")
print("=" * 55)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Số lượng': missing, 'Tỷ lệ (%)': missing_pct})
missing_df = missing_df[missing_df['Số lượng'] > 0].sort_values('Tỷ lệ (%)', ascending=False)
if missing_df.empty:
    print("✅ Không có missing values!")
else:
    print(missing_df.to_string())
    print()
    print("→ Chiến lược xử lý avg_rating: impute bằng MEDIAN của tập TRAIN (sau split)")
    print("  (Ghi chú: ở bước này impute tạm bằng median toàn bộ — thống nhất với NB3)")

# === 3.2: Kiểm tra Duplicates ===
print()
print("=" * 55)
print("3.2 DUPLICATES")
print("=" * 55)
n_dup = df.duplicated().sum()
print(f"Số dòng trùng lặp: {n_dup:,}")
if n_dup > 0:
    df = df.drop_duplicates()
    print(f"→ Đã xóa → còn lại {len(df):,} dòng")
else:
    print("✅ Không có dòng trùng lặp")

# === 3.3: Impute avg_rating ===
print()
print("=" * 55)
print("3.3 IMPUTE avg_rating")
print("=" * 55)
avg_rating_median = df['avg_rating'].median()
n_nan_rating = df['avg_rating'].isna().sum()
df['avg_rating'] = df['avg_rating'].fillna(avg_rating_median)
print(f"Đã fill {n_nan_rating:,} NaN ← median = {avg_rating_median:.3f}")
print(f"✅ avg_rating: không còn NaN ({df['avg_rating'].isna().sum()})")


In [ ]:

# === 3.4: Phân phối Target và Correlation ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Bước 3 — Phân tích dữ liệu tổng quan', fontsize=14, fontweight='bold', y=1.02)

# --- (A) Target Distribution ---
counts = df['repurchase'].value_counts().sort_index()
colors = ['#DC2626', '#059669']
bars = axes[0].bar(['Không tái mua (0)', 'Tái mua (1)'], counts.values,
                   color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{cnt:,}\n({cnt/len(df)*100:.1f}%)', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')
axes[0].set_title('Phân phối Target (repurchase)', fontsize=13)
axes[0].set_ylabel('Số lượng')
axes[0].set_ylim(0, max(counts.values) * 1.22)

# --- (B) Missing values bar ---
miss_pct = df.isnull().mean() * 100
miss_pct = miss_pct[miss_pct > 0]
if miss_pct.empty:
    axes[1].text(0.5, 0.5, '✅ Không có\nMissing Values', ha='center', va='center',
                 fontsize=15, color='#059669', fontweight='bold', transform=axes[1].transAxes)
    axes[1].set_title('Missing Values', fontsize=13)
    axes[1].axis('off')
else:
    miss_pct.plot(kind='barh', ax=axes[1], color='#F59E0B')
    axes[1].set_title('Missing Values (%)', fontsize=13)
    axes[1].set_xlabel('Tỷ lệ (%)')

# --- (C) Numeric features correlation heatmap (top 10) ---
num_cols = df.select_dtypes(include=np.number).drop(columns=['repurchase'], errors='ignore')
corr_with_target = abs(df[num_cols.columns].corrwith(df['repurchase'])).sort_values(ascending=False)
top_cols = corr_with_target.head(10).index.tolist()
corr_matrix = df[top_cols + ['repurchase']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, ax=axes[2], cmap='coolwarm', center=0, annot=True,
            fmt='.2f', linewidths=0.5, annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8})
axes[2].set_title('Correlation Heat-map\n(Top 10 numeric features)', fontsize=13)
axes[2].tick_params(axis='x', rotation=45, labelsize=8)
axes[2].tick_params(axis='y', rotation=0, labelsize=8)

plt.tight_layout()
plt.show()
print(f"\n✅ Dataset sạch: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"   Class imbalance: 0={counts[0]:,} ({counts[0]/len(df)*100:.1f}%) | 1={counts[1]:,} ({counts[1]/len(df)*100:.1f}%)")



---
# Bước 4: Định nghĩa Label / Target

**Label:** `repurchase`
- `0` = Không tái mua (khách hàng mua ≤ 4 lần → churned)
- `1` = Tái mua (khách hàng mua ≥ 5 lần → loyal)

**Ngưỡng xác định:** Otsu thresholding trên `purchase_count` → T* = 4.02 (tự động, không bias)

**Cột bị loại trước khi tạo X:**
| Cột | Lý do |
|-----|-------|
| `user_id` | Định danh — không có predictive value |
| `product_category` | High-cardinality, cần target encoding nâng cao |
| `repurchase` | Chính là target y |
| `total_interactions` | ⚠️ Đã loại ở bước 2 (leaky) |

**Categorical encoding (thực hiện trước khi tạo X):**
- `dominant_gender` → LabelEncoder (M/F/Other → 0/1/2)
- `dominant_location` → LabelEncoder (tên thành phố → int)


In [ ]:

# === BƯỚC 4: Encode categorical + Tách X / y ===

# --- Encode categorical features ---
le_gender   = LabelEncoder()
le_location = LabelEncoder()
df['dominant_gender']   = le_gender.fit_transform(df['dominant_gender'].astype(str))
df['dominant_location'] = le_location.fit_transform(df['dominant_location'].astype(str))
print("✅ Categorical encoded:")
print(f"   dominant_gender   → classes: {le_gender.classes_}")
print(f"   dominant_location → {len(le_location.classes_)} classes")

# --- Loại cột không phải features ---
DROP_COLS = ['user_id', 'product_category', 'repurchase']
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# --- Định nghĩa features RAW ---
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f"\n✅ Raw features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

y = df['repurchase'].values
X_raw = df[FEATURE_COLS].copy()

print(f"\n   X shape: {X_raw.shape}  |  y shape: {y.shape}")
print(f"   Tỷ lệ tái mua: {y.mean()*100:.1f}%  |  scale_pos_weight ≈ {(1-y.mean())/y.mean():.3f}")



---
# Bước 5: Feature Engineering — Tạo Features Mới

**Chiến lược:** Từ 16 raw features → tạo thêm **15 derived features** (7 features cũ từ NB3 + 8 features mới)

**Nguyên tắc chống leakage:**
- Chỉ dùng raw behavioral / aggregate features ✅
- KHÔNG dùng `total_interactions` (chứa purchase_count) ❌
- KHÔNG dùng `purchase_count` (đã xóa khỏi dataset) ❌
- `pure_beh = view + click + cart + wishlist` — an toàn ✅

| # | Feature | Công thức | Ý nghĩa |
|:--:|---------|-----------|---------|
| **7 FEATURES CŨ (từ NB3)** ||||
| 1 | view_to_click_ratio | total_click / (total_view + 1) | Tỷ lệ click từ lượt xem |
| 2 | cart_to_click_ratio | total_cart / (total_click + 1) | Tỷ lệ thêm giỏ từ click |
| 3 | wishlist_to_view_ratio | total_wishlist / (total_view + 1) | Lưu yêu thích từ lượt xem |
| 4 | active_engagement_ratio | (click + cart) / (pure_beh + 1) | Mức độ tương tác chủ động |
| 5 | category_commitment_score | category_share × pure_beh | Gắn bó category |
| 6 | exploration_score | unique_brands / (total_cart + 1) | Mức độ khám phá brand mới |
| 7 | engagement_depth_score | pure_beh / (user_total_categories + 1) | Độ sâu tương tác mỗi category |
| **8 FEATURES MỚI** ||||
| 8 | discount_sensitivity | avg_discount / (avg_price + 1e-6) | % giảm giá thực hưởng — cao = nhạy với discount |
| 9 | brand_loyalty_score | total_cart / (unique_brands + 1) | Cart tập trung vào ít brand — cao = trung thành brand |
| 10 | wishlist_to_cart_ratio | total_wishlist / (total_cart + 1) | Wishlist nhiều hơn cart → "xem nhiều mua ít" |
| 11 | click_engagement_rate | (total_click + total_cart) / (total_view + 1) | % xem dẫn đến tương tác sâu |
| 12 | value_per_click | avg_purchase_value / (total_click + 1) | Giá trị mua trên mỗi lần click |
| 13 | price_to_value_ratio | avg_purchase_value / (avg_price + 1e-6) | % giá gốc thực trả (gần 1 = ít discount) |
| 14 | category_breadth_score | user_total_categories / (unique_brands + 1) | Đa dạng category trên mỗi brand |
| 15 | rating_concentration | avg_rating × category_share | Hài lòng cao trong category tập trung |

**Tổng:** 16 raw + 15 derived = **31 features** trước Feature Selection


In [ ]:

# === BƯỚC 5: Feature Engineering — Hàm tạo features ===

def add_derived_features(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo 15 derived features từ raw features.
    Chỉ dùng raw behavioral/aggregate features — KHÔNG có leakage.
    
    Tham số: df_in — DataFrame với các raw features (không có total_interactions)
    Trả về : df_in với 15 cột mới được thêm vào
    """
    df_out = df_in.copy()
    
    # Pure behavioral sum (an toàn — không chứa purchase_count)
    pure_beh = (df_out['total_view'] + df_out['total_click'] +
                df_out['total_cart'] + df_out['total_wishlist'])
    
    # ── 7 FEATURES CŨ (từ NB3 — đã được validate) ─────────────────────────────
    df_out['view_to_click_ratio']        = df_out['total_click'] / (df_out['total_view'] + 1)
    df_out['cart_to_click_ratio']        = df_out['total_cart']  / (df_out['total_click'] + 1)
    df_out['wishlist_to_view_ratio']     = df_out['total_wishlist'] / (df_out['total_view'] + 1)
    df_out['active_engagement_ratio']    = (df_out['total_click'] + df_out['total_cart']) / (pure_beh + 1)
    df_out['category_commitment_score']  = df_out['category_share'] * pure_beh
    df_out['exploration_score']          = df_out['unique_brands'] / (df_out['total_cart'] + 1)
    df_out['engagement_depth_score']     = pure_beh / (df_out['user_total_categories'] + 1)
    
    # ── 8 FEATURES MỚI ────────────────────────────────────────────────────────
    # 8. % giảm giá thực hưởng trên giá gốc
    df_out['discount_sensitivity']   = df_out['avg_discount'] / (df_out['avg_price'] + 1e-6)
    
    # 9. Cart tập trung trên ít brand → trung thành brand cao
    df_out['brand_loyalty_score']    = df_out['total_cart'] / (df_out['unique_brands'] + 1)
    
    # 10. Wishlist nhiều hơn cart → "window shopper", ít quyết tâm mua
    df_out['wishlist_to_cart_ratio'] = df_out['total_wishlist'] / (df_out['total_cart'] + 1)
    
    # 11. % lượt xem dẫn đến tương tác sâu (click/cart)
    df_out['click_engagement_rate']  = (df_out['total_click'] + df_out['total_cart']) / (df_out['total_view'] + 1)
    
    # 12. Giá trị trung bình mua hàng mỗi lần click
    df_out['value_per_click']        = df_out['avg_purchase_value'] / (df_out['total_click'] + 1)
    
    # 13. Tỷ lệ giá thực trả / giá gốc (gần 1 = ít discount, <1 = hay giảm giá nhiều)
    df_out['price_to_value_ratio']   = df_out['avg_purchase_value'] / (df_out['avg_price'] + 1e-6)
    
    # 14. Số category trên mỗi brand (cao = khách đa dạng, khám phá nhiều category)
    df_out['category_breadth_score'] = df_out['user_total_categories'] / (df_out['unique_brands'] + 1)
    
    # 15. Rating × category_share: hài lòng cao trong category tập trung → gắn bó mạnh
    df_out['rating_concentration']   = df_out['avg_rating'] * df_out['category_share']
    
    return df_out


# Áp dụng cho toàn bộ dataset
X_engineered = add_derived_features(X_raw)

n_raw = len(FEATURE_COLS)
n_derived = 15
n_total = len(X_engineered.columns)

print(f"✅ Feature Engineering hoàn thành!")
print(f"   Raw features    : {n_raw}")
print(f"   Derived features: {n_derived}")
print(f"   Tổng            : {n_total}")
print()
print("Các features mới (cuối cùng trong DataFrame):")
new_cols = X_engineered.columns[n_raw:]
for i, col in enumerate(new_cols, 1):
    print(f"   {i:2d}. {col}")


In [ ]:

# === BƯỚC 5 (tiếp): Visualize phân phối 8 features mới ===
new_derived = [
    'discount_sensitivity', 'brand_loyalty_score', 'wishlist_to_cart_ratio',
    'click_engagement_rate', 'value_per_click', 'price_to_value_ratio',
    'category_breadth_score', 'rating_concentration'
]

df_vis = X_engineered[new_derived].copy()
df_vis['repurchase'] = y

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Phân phối 8 Features Mới (theo nhãn tái mua)', fontsize=14, fontweight='bold')

colors = {0: '#DC2626', 1: '#059669'}
labels_map = {0: 'Không tái mua', 1: 'Tái mua'}

for idx, col in enumerate(new_derived):
    ax = axes[idx // 4][idx % 4]
    for label in [0, 1]:
        vals = df_vis.loc[df_vis['repurchase'] == label, col]
        ax.hist(vals, bins=40, alpha=0.6, color=colors[label],
                label=labels_map[label], density=True)
    ax.set_title(col.replace('_', '\n'), fontsize=9, pad=4)
    ax.set_xlabel('')
    ax.tick_params(labelsize=8)
    if idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("✅ 8 features mới — phân phối rõ ràng theo nhãn = có predictive value!")



---
# Bước 6: Feature Selection — Lọc correlation cao

**Lý do:** Các features có correlation > 0.9 với nhau mang thông tin trùng lặp → gây **multicollinearity** (hại cho Logistic Regression), tăng complexity vô ích.

**Phương pháp:** Correlation filter — giữ feature đầu tiên trong cặp có |corr| > 0.9, loại feature thứ hai.

> **Ghi chú:** Đây là kỹ thuật đơn giản phù hợp cho prototype. Trong production có thể dùng VIF (Variance Inflation Factor) hoặc PCA.


In [ ]:

# === BƯỚC 6: Feature Selection — Correlation Filter ===
CORR_THRESHOLD = 0.9

corr_matrix = X_engineered.corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Tìm cột có correlation > threshold với cột khác (giữ cột đầu, loại cột sau)
to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > CORR_THRESHOLD)]
X_selected = X_engineered.drop(columns=to_drop)

print(f"Correlation threshold: {CORR_THRESHOLD}")
print(f"Trước selection: {X_engineered.shape[1]} features")
print(f"Loại bỏ ({len(to_drop)}): {to_drop}")
print(f"Sau selection : {X_selected.shape[1]} features")
print()

# Vẽ correlation heatmap sau selection
fig, ax = plt.subplots(figsize=(16, 13))
corr_after = X_selected.corr()
mask = np.triu(np.ones_like(corr_after, dtype=bool))
sns.heatmap(corr_after, mask=mask, cmap='coolwarm', center=0, annot=True,
            fmt='.2f', linewidths=0.3, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title(f'Correlation Matrix sau Feature Selection ({X_selected.shape[1]} features)',
             fontsize=13, fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.show()
print(f"✅ Feature Selection hoàn thành: {X_selected.shape[1]} features giữ lại")



---
# Bước 7: Chia dữ liệu Train / Test + Chuẩn hóa

**Chiến lược:**
- **80/20 split** với `stratify=y` → giữ tỷ lệ class imbalance đồng đều ở train và test
- **StandardScaler** chỉ fit trên tập TRAIN — tránh data leakage từ test set vào scaler
- Scaler chỉ cần cho **Logistic Regression** (nhạy với scale) — RF và XGBoost không cần

> ⚠️ Quy tắc vàng: **fit() chỉ trên train, transform() trên cả train và test**


In [ ]:

# === BƯỚC 7: Train/Test Split + Scaling ===
X = X_selected.values
feature_names = X_selected.columns.tolist()

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"✅ Train/Test split (80/20, stratified):")
print(f"   Train: {X_train.shape[0]:,} samples  | Test: {X_test.shape[0]:,} samples")
print(f"   Train label=1: {y_train.mean()*100:.1f}%  | Test label=1: {y_test.mean()*100:.1f}%")

# StandardScaler — fit ONLY on train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)        # chỉ transform (không fit)

print(f"\n✅ StandardScaler (cho Logistic Regression):")
print(f"   X_train_scaled mean ≈ {X_train_scaled.mean():.4f}  std ≈ {X_train_scaled.std():.4f}")
print(f"   X_test_scaled  mean ≈ {X_test_scaled.mean():.4f}  std ≈ {X_test_scaled.std():.4f}")

# Tính scale_pos_weight cho XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\n   scale_pos_weight (XGBoost) = {scale_pos_weight:.3f}")



---
# Bước 8 + 9: Chọn và Huấn luyện Mô hình

**3 mô hình baseline:**
| Model | Strengths | Lưu ý |
|-------|-----------|-------|
| **Logistic Regression** | Nhanh, interpretable, baseline tốt | Cần StandardScaler, giới hạn với non-linear patterns |
| **Random Forest** | Robust, handle categorical, feature importance | Chậm hơn, cần tune n_estimators |
| **XGBoost** | Thường tốt nhất cho tabular, handle imbalance tốt | Cần `scale_pos_weight` cho imbalanced data |

**Xử lý Class Imbalance:**
- LR: `class_weight='balanced'`
- RF: `class_weight='balanced'`
- XGB: `scale_pos_weight = n_negative / n_positive ≈ 2.05`


In [ ]:

# === BƯỚC 8+9: Huấn luyện 3 mô hình ===

# --- Logistic Regression (dùng scaled data) ---
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)
print("✅ Logistic Regression — trained")

# --- Random Forest (dùng unscaled data) ---
rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
print("✅ Random Forest      — trained")

# --- XGBoost (dùng unscaled data) ---
xgb = XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0
)
xgb.fit(X_train, y_train)
print("✅ XGBoost            — trained")

print("\nTất cả 3 mô hình đã được huấn luyện thành công!")



---
# Bước 10: Đánh giá mô hình

**Metrics sử dụng:**
- **Accuracy**: % dự đoán đúng tổng thể (ít ý nghĩa với imbalanced data)
- **AUC**: Khả năng phân biệt 2 class (1.0 = hoàn hảo)
- **Precision**: Trong số những người dự đoán là tái mua, bao nhiêu % đúng
- **Recall**: Trong số người thực sự tái mua, model bắt được bao nhiêu %
- **F1**: Harmonic mean của Precision và Recall → metric chính

**KPI mục tiêu:** F1 ≥ 0.72, AUC ≥ 0.87


In [ ]:

# === BƯỚC 10: Đánh giá tất cả mô hình ===

def evaluate_model(model, X_eval, y_eval, model_name='Model', use_scaled=False, scaler_=None):
    """Tính đầy đủ metrics cho một model."""
    X_ = scaler_.transform(X_eval) if use_scaled and scaler_ else X_eval
    y_pred  = model.predict(X_)
    y_proba = model.predict_proba(X_)[:, 1]   # P(repurchase=1)
    return {
        'Model'    : model_name,
        'Accuracy' : accuracy_score(y_eval, y_pred),
        'AUC'      : roc_auc_score(y_eval, y_proba),
        'Precision': precision_score(y_eval, y_pred, pos_label=1, zero_division=0),
        'Recall'   : recall_score(y_eval, y_pred, pos_label=1, zero_division=0),
        'F1'       : f1_score(y_eval, y_pred, pos_label=1, zero_division=0),
        '_proba'   : y_proba,
        '_pred'    : y_pred,
    }

# Đánh giá
results_raw = [
    evaluate_model(lr,  X_test, y_test, 'Logistic Regression', use_scaled=True, scaler_=scaler),
    evaluate_model(rf,  X_test, y_test, 'Random Forest'),
    evaluate_model(xgb, X_test, y_test, 'XGBoost'),
]

# Bảng tổng hợp
results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results_raw])
results_df = results_df.set_index('Model')

print("=" * 65)
print("BẢNG SO SÁNH HIỆU NĂNG — BASELINE MODELS")
print("=" * 65)
print(results_df.to_string())
print("=" * 65)
print(f"\nKPI mục tiêu: F1 ≥ 0.72 | AUC ≥ 0.87")
for r in results_raw:
    f1_ok  = "✅" if r['F1']  >= 0.72 else "❌"
    auc_ok = "✅" if r['AUC'] >= 0.87 else "❌"
    print(f"  {r['Model']:<22} F1={r['F1']:.4f} {f1_ok}  AUC={r['AUC']:.4f} {auc_ok}")


In [ ]:

# === BƯỚC 10 (tiếp): ROC Curves + Feature Importance ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Bước 10 — Đánh giá Mô hình Baseline', fontsize=14, fontweight='bold')

# --- (A) ROC Curves ---
model_colors = {'Logistic Regression': '#2563EB', 'Random Forest': '#059669', 'XGBoost': '#DC2626'}
for r in results_raw:
    fpr, tpr, _ = roc_curve(y_test, r['_proba'], pos_label=1)
    axes[0].plot(fpr, tpr, color=model_colors[r['Model']], lw=2,
                 label=f"{r['Model']} (AUC={r['AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].axhline(y=0.87, color='#F59E0B', linestyle=':', lw=1.5, alpha=0.8, label='KPI AUC=0.87')
axes[0].fill_between([0, 1], [0.87, 0.87], 0, alpha=0.03, color='#F59E0B')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curves — 3 Mô hình Baseline', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_xlim(-0.02, 1.02)
axes[0].set_ylim(-0.02, 1.05)

# --- (B) Random Forest Feature Importance (Top 20) ---
fi = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
fi_top20 = fi.tail(20)
colors_fi = ['#059669' if i >= len(fi_top20) - 5 else '#2563EB' for i in range(len(fi_top20))]
fi_top20.plot(kind='barh', ax=axes[1], color=colors_fi, edgecolor='white', linewidth=0.5)
axes[1].set_title('Random Forest — Top 20 Feature Importance', fontsize=12)
axes[1].set_xlabel('Importance Score', fontsize=11)
axes[1].tick_params(axis='y', labelsize=9)
# Đánh dấu features mới
new_derived_set = set([
    'discount_sensitivity', 'brand_loyalty_score', 'wishlist_to_cart_ratio',
    'click_engagement_rate', 'value_per_click', 'price_to_value_ratio',
    'category_breadth_score', 'rating_concentration'
])
for tick in axes[1].get_yticklabels():
    if tick.get_text() in new_derived_set:
        tick.set_color('#059669')
        tick.set_fontweight('bold')

plt.tight_layout()
plt.show()
print("✅ ROC curves và Feature Importance đã render!")
print("   (Label màu xanh lá = features mới được thêm)")



---
# Bước 11: Giảm chiều Feature — Feature Reduction

**Lý do:** Sau khi có feature importance từ Random Forest, loại bỏ các features có importance < 0.01 → mô hình gọn hơn, ít overfitting hơn, inference nhanh hơn.

> Đây là bước **tùy chọn** — thực tế chỉ làm khi số feature lớn và có nhiều feature importance gần 0.


In [ ]:

# === BƯỚC 11: Feature Reduction (importance threshold) ===
IMPORTANCE_THRESHOLD = 0.01

fi_all = pd.Series(rf.feature_importances_, index=feature_names)
important_features = fi_all[fi_all >= IMPORTANCE_THRESHOLD].index.tolist()
dropped_features   = fi_all[fi_all <  IMPORTANCE_THRESHOLD].index.tolist()

print(f"Importance threshold: {IMPORTANCE_THRESHOLD}")
print(f"Trước reduction: {len(feature_names)} features")
print(f"Loại bỏ ({len(dropped_features)}): {dropped_features}")
print(f"Giữ lại  ({len(important_features)}): {important_features}")

# Lọc lại train / test
feat_idx   = [feature_names.index(f) for f in important_features]
X_train_r  = X_train[:, feat_idx]
X_test_r   = X_test[:, feat_idx]

# Scaled version cho LR
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled  = scaler_r.transform(X_test_r)

print(f"\n✅ Feature Reduction: X_train {X_train.shape[1]} → {X_train_r.shape[1]} features")



---
# Bước 12: Tinh chỉnh Hyperparameter — GridSearchCV

**Chiến lược:** Tinh chỉnh **Random Forest** với GridSearchCV và StratifiedKFold (5 fold) để:
1. Tìm `n_estimators` tốt nhất (100, 200, 500)
2. Tìm `max_depth` tốt nhất (None, 5, 10, 20)
3. Tìm `min_samples_split` tốt nhất (2, 5, 10)

**Scoring:** `f1` — phù hợp với bài toán imbalanced classification (không dùng accuracy)

> ⏱️ GridSearch với RF thường mất 2-5 phút — bình thường!


In [ ]:

# === BƯỚC 12: GridSearchCV — Tinh chỉnh Random Forest ===
import time

param_grid = {
    'n_estimators'    : [100, 200, 500],
    'max_depth'       : [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_base = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)

grid_search = GridSearchCV(
    rf_base, param_grid,
    cv=cv, scoring='f1', n_jobs=-1, verbose=0
)

print("GridSearchCV đang chạy... (có thể mất vài phút)")
t0 = time.time()
grid_search.fit(X_train_r, y_train)
elapsed = time.time() - t0

print(f"\n✅ GridSearchCV hoàn thành trong {elapsed:.1f}s")
print(f"   Best params : {grid_search.best_params_}")
print(f"   Best CV F1  : {grid_search.best_score_:.4f}")



---
# Bước 13: Retrain với Best Parameters

Sau khi GridSearchCV tìm ra best params, **retrain toàn bộ tập train** với các params đó (không chỉ trên CV fold).


In [ ]:

# === BƯỚC 13: Retrain với best params ===
best_params = grid_search.best_params_

rf_best = RandomForestClassifier(
    **best_params,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_best.fit(X_train_r, y_train)

# Đánh giá sau retrain
y_pred_best  = rf_best.predict(X_test_r)
y_proba_best = rf_best.predict_proba(X_test_r)[:, 1]

f1_best  = f1_score(y_test, y_pred_best, pos_label=1)
auc_best = roc_auc_score(y_test, y_proba_best)
prec_best = precision_score(y_test, y_pred_best, pos_label=1)
rec_best  = recall_score(y_test, y_pred_best, pos_label=1)

print("=" * 55)
print("BƯỚC 13 — RF Best (sau GridSearchCV + Retrain)")
print("=" * 55)
print(f"  Best params: {best_params}")
print()
print(f"  F1        : {f1_best:.4f}")
print(f"  AUC       : {auc_best:.4f}")
print(f"  Precision : {prec_best:.4f}")
print(f"  Recall    : {rec_best:.4f}")
print("=" * 55)



---
# Bước 14: Kiểm tra KPI — Vòng lặp đánh giá

**Logic KPI Check:**
- Nếu cả 2 điều kiện thỏa mãn → Model đạt yêu cầu → deploy
- Nếu thiếu → phân tích nguyên nhân → điều chỉnh (thêm features, thay model, tune params)

**KPI mục tiêu:**
- F1 ≥ 0.72 → Cân bằng Precision/Recall tốt cho CRM targeting
- AUC ≥ 0.87 → Khả năng phân biệt class tốt (discriminative power)


In [ ]:

# === BƯỚC 14: KPI Check ===
KPI_F1  = 0.72
KPI_AUC = 0.87

# So sánh tất cả models
all_results = [
    *results_raw,
    {
        'Model': 'RF Best (GridSearch)',
        'F1'   : f1_best,
        'AUC'  : auc_best,
        'Precision': prec_best,
        'Recall'   : rec_best,
        '_proba'   : y_proba_best,
        '_pred'    : y_pred_best,
    }
]

print("=" * 70)
print("BƯỚC 14 — KPI CHECK")
print("=" * 70)
print(f"{'Model':<28} {'F1':>8} {'F1 KPI':>8} {'AUC':>8} {'AUC KPI':>9}")
print("-" * 70)

best_model_obj  = None
best_model_name = None
best_f1_score   = 0

for r in all_results:
    f1_pass  = r['F1']  >= KPI_F1
    auc_pass = r['AUC'] >= KPI_AUC
    f1_icon  = "✅" if f1_pass  else "❌"
    auc_icon = "✅" if auc_pass else "❌"
    both_pass = "⭐ PASS" if (f1_pass and auc_pass) else "   FAIL"
    print(f"  {r['Model']:<26} {r['F1']:>7.4f} {f1_icon}  {r['AUC']:>7.4f} {auc_icon}  {both_pass}")
    
    if r['F1'] > best_f1_score:
        best_f1_score   = r['F1']
        best_model_name = r['Model']

print("=" * 70)
print(f"\nModel tốt nhất (F1): {best_model_name} (F1={best_f1_score:.4f})")
print()

# Chọn best final model
if f1_best >= KPI_F1 and auc_best >= KPI_AUC:
    print("✅ RF Best đạt KPI → Dùng RF Best cho Bước 15")
    final_model   = rf_best
    final_X_train = X_train_r
    final_X_test  = X_test_r
    final_proba   = y_proba_best
else:
    # Fallback: dùng XGBoost
    r_xgb = results_raw[2]
    print(f"❌ RF Best chưa đạt KPI → Fallback: XGBoost (F1={r_xgb['F1']:.4f})")
    final_model   = xgb
    final_X_train = X_train
    final_X_test  = X_test
    final_proba   = r_xgb['_proba']



---
# Bước 15: Tối ưu hóa Ngưỡng Phân loại (Cutoff Threshold)

**Vấn đề:** Model mặc định dùng ngưỡng **threshold = 0.5** → không phải lúc nào cũng tối ưu!

**Tại sao cần điều chỉnh threshold?**
- Threshold thấp (vd 0.3): Recall cao hơn → bắt được nhiều khách tái mua hơn → nhưng gửi nhiều voucher nhầm
- Threshold cao (vd 0.7): Precision cao hơn → ít gửi nhầm → nhưng bỏ sót nhiều khách tiềm năng

**Trong bài toán CRM:**
- Ưu tiên cân bằng F1 → tìm threshold maximize F1
- Hoặc nếu có ngân sách voucher cố định → tìm threshold với Recall ≥ 0.70

**Phương pháp:** Vẽ đường cong F1 / Precision / Recall theo threshold → chọn điểm tối ưu


In [ ]:

# === BƯỚC 15: Tìm Optimal Threshold ===
thresholds   = np.linspace(0.01, 0.99, 200)
f1_scores    = []
prec_scores  = []
rec_scores   = []

for t in thresholds:
    y_pred_t = (final_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t, pos_label=1, zero_division=0))
    prec_scores.append(precision_score(y_test, y_pred_t, pos_label=1, zero_division=0))
    rec_scores.append(recall_score(y_test, y_pred_t, pos_label=1, zero_division=0))

# Tìm ngưỡng tối ưu (maximize F1)
best_idx       = np.argmax(f1_scores)
optimal_thresh = thresholds[best_idx]
optimal_f1     = f1_scores[best_idx]
optimal_prec   = prec_scores[best_idx]
optimal_rec    = rec_scores[best_idx]

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Bước 15 — Tối ưu hóa Threshold Phân loại', fontsize=14, fontweight='bold')

# (A) Threshold curves
axes[0].plot(thresholds, f1_scores,   label='F1',        color='#2563EB', lw=2)
axes[0].plot(thresholds, prec_scores, label='Precision',  color='#059669', lw=1.5, linestyle='--')
axes[0].plot(thresholds, rec_scores,  label='Recall',     color='#DC2626', lw=1.5, linestyle='--')
axes[0].axvline(x=optimal_thresh, color='#F59E0B', linestyle=':', lw=2,
                label=f'Optimal={optimal_thresh:.3f}')
axes[0].axvline(x=0.5, color='#94A3B8', linestyle=':', lw=1.5, label='Default=0.5')
axes[0].scatter([optimal_thresh], [optimal_f1], s=100, color='#F59E0B', zorder=5)
axes[0].set_xlabel('Threshold', fontsize=11)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('F1 / Precision / Recall theo Threshold', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# (B) Confusion Matrix tại optimal threshold
y_pred_opt = (final_proba >= optimal_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)
labels_cm = ['Không tái mua\n(0)', 'Tái mua\n(1)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=labels_cm, yticklabels=labels_cm,
            linewidths=1, cbar_kws={'shrink': 0.8})
axes[1].set_title(f'Confusion Matrix — Threshold={optimal_thresh:.3f}', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('Actual', fontsize=11)

plt.tight_layout()
plt.show()

print("=" * 55)
print("BƯỚC 15 — KẾT QUẢ CUỐI CÙNG (optimal threshold)")
print("=" * 55)
print(f"  Default threshold (0.50) :")
y_pred_default = (final_proba >= 0.50).astype(int)
f1_default  = f1_score(y_test, y_pred_default, pos_label=1)
rec_default = recall_score(y_test, y_pred_default, pos_label=1)
print(f"    F1={f1_default:.4f}  Recall={rec_default:.4f}")
print()
print(f"  Optimal threshold ({optimal_thresh:.3f}):")
print(f"    F1={optimal_f1:.4f}  Precision={optimal_prec:.4f}  Recall={optimal_rec:.4f}")
print("=" * 55)
improvement = (optimal_f1 - f1_default) / f1_default * 100
print(f"  → Cải thiện F1: +{improvement:.1f}% nhờ threshold tuning")



---
# Tổng kết — 15 Bước ML Pipeline Hoàn Chỉnh

## Kết quả đạt được

| Bước | Kết quả |
|------|---------|
| 1. Đặt vấn đề | KPI: F1 ≥ 0.72, AUC ≥ 0.87 |
| 2. Dữ liệu | 61,728 mẫu × 16 raw features |
| 3. Tiền xử lý | avg_rating imputed, no duplicates |
| 4. Target | repurchase binary, imbalance 67/33% |
| 5. Feature Engineering | **31 features** (16 raw + 15 derived: 7 cũ + 8 mới) |
| 6. Feature Selection | Lọc correlation > 0.9 |
| 7. Split + Scale | 80/20 stratified, StandardScaler |
| 8+9. Models | LR / Random Forest / XGBoost |
| 10. Evaluation | So sánh F1, AUC, ROC curves |
| 11. Reduction | Loại features importance < 0.01 |
| 12. GridSearch | RandomForest hyperparameter tuning |
| 13. Retrain | Best params → full train |
| 14. KPI Check | Kiểm tra F1 ≥ 0.72 & AUC ≥ 0.87 |
| 15. Threshold | Optimal cutoff → maximize F1 |

## 8 Features Mới — Đóng góp

| Feature mới | Ý nghĩa business |
|-------------|-----------------|
| `discount_sensitivity` | Khách nhạy với giảm giá vs khách trung thành |
| `brand_loyalty_score` | Tập trung cart vào ít brand → trung thành hơn |
| `wishlist_to_cart_ratio` | "Window shopper" index — cao = xem không mua |
| `click_engagement_rate` | % xem → tương tác sâu (click/cart) |
| `value_per_click` | Giá trị mua trên mỗi lần click |
| `price_to_value_ratio` | Tỷ lệ trả đủ giá vs dùng discount |
| `category_breadth_score` | Đa dạng category trên mỗi brand |
| `rating_concentration` | Hài lòng × gắn bó category |

---
> 💡 **Ghi nhớ cuối:** Pipeline này là template. Trong production, mỗi bước cần logging, versioning (MLflow), và A/B testing trước khi deploy.
